In [1]:
 #i/o imports, pandas
from pathlib import Path
from typing import List, Tuple
import pandas as pd

#rdkit imports
import rdkit
from rdkit import Chem
# from rdkit.Chem import AllChem
# from rdkit.Chem import Draw
# from itertools import chain


"""
### 'Prepare_Arene_Cores.ipynb' - @GCH v1.0 - last updt. 04/20/2026

### Notebook Overview:
This notebook contains code to pre-parse incoming raw SMILES data to ensure all incoming SMILES data are valid and chemically 
relevant. In simpler words, this is a project-specific Notebook to ensure we have a self-consistent set of arene cores suitable
for conversion to arynes via Reaction SMARTS. Cells in this notebook rely on SMARTS-based substructure searches/skeletal 
editing to generate a set of unique arene cores that contain at least one C=C double bond convertible to a C#C triple bond.
Following rough skeletal editing to produce the maximal set of possible [hetero]arene precursors, we next remove any/all 
remaining exocylic substituents to consider the aromatic core, only. A final deduplication pass results in a set of unique and
aryne-convertible arene precurors in SMILES format. 

### Specifically, the following checks/edits are performed on each incoming SMILES string:
1. Attempt to generate a valid/sanitized RDKit Mol object; drop invalid mols
   
2. Apply SMARTS-based substructure search for exocylic -Ph substituents; replace ring-Ph with ring-CH3 (R-Me). Deduplicate.

3. Apply SMARTS-based substructure search for vinylic substituents ring-C=C-R; Fragment to generate ring-CH3 and HC-R
   (and any additional fragments if mutiple vinylic couples). Search for and retain fragments with aromatic ring
   subunits. Deduplicate.

4. Apply SMARTS-based substructure search for biaryl subunits (ex: Ar-Ar); Fragment at the biaryl sigma bond indices to
   generate distinct ring systems; deduplicate

5. Perform an explicit check for the presence of at least one C=C double bond within the ring that is convertible to
    a C#C triple bond

### Motivation:
A mandatory and thorough pre-parsing of SMILES data to ensure that we are only considering valid heterocycles in our database.

### How to Use this Notebook: 
1. Define relevant Paths to output files in the PATH cell below
2. Run all cells in the notebook

"""

In [2]:
### Define PATHS/relevant directories ### 

module2_dir = Path.cwd() #path containing the current script (Module2 dir)

#the project TLD is 1 "parent" above current dir
project_tld = module2_dir.parent

#The Module1 directory contains the .csv we need in this notebook
input_csv_dir = project_tld / "Module1_Merge_CSVs_Deduplicate_SMILES" / "combined_and_deduplicated_smiles.csv"

#Read in the specified .csv as a pd DataFrame
incoming_data = pd.read_csv(input_csv_dir)
incoming_data.head() #print to console

,data_source,regid,smiles
0,C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_L...,arbr141,Cc1[nH]nc(C(F)(F)F)c1Br
1,C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_L...,arbr142,Cc1[nH]nc(-c2ccccc2)c1Br
2,C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_L...,arbr143,Cc1cc(C(C)(C)C)cc(C)c1Br
3,C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_L...,arbr145,Cc1cccc(Cl)c1Br
4,C:\Chem_Work\CompChem_Working\CCAS_Hetarynes_L...,arbr146,Cc1cccc(F)c1Br


In [3]:
def smiles_to_mols_rdkit(list_of_smiles: List[str], sanitization=True) -> List[Chem.Mol]:
    """
    Function: Converts an SD-file containing a single molecule to an RDKit mol object. Returns the mol object. 
    
    Input:
        - list_of_smiles: List[str]; a list of raw SMILES strings to convert to Mol objects
        - sanitization = True (default val True) - use RDKIT sanitization on the SMILES str; good/safe practice
        
    Returns:
        - list_of_valid_mols: List[Chem.Mol]; a list of validated RDKit Mol objects
    """

    #init a list to store validated RDKit mol objects
    list_of_valid_mols = []
    #Init a counter to keep track of any failed SMILES => Mol conversions
    failed_count = 0

    #loop over each of the raw SMILES strings in the incoming list    
    #Using 'enumerate' gives us access to the index of each SMILES in the list
    for index, smile in enumerate(list_of_smiles):
        
        try: #Attempt to convert the SMILES string to RDKit Mol Object 
            #here we attempt the conversion and explicitly require sanitization
            mol = Chem.MolFromSmiles(smile, sanitize = sanitization)

            if mol is not None: #If the mol is convertible (not NONE type), 
                #append the mol to our list of validated mols
                list_of_valid_mols.append(mol)

            else: #if the SMILE cant be converted, 
                failed_count += 1 #iterate our failed counter
                continue
                
        #some SMILES may be unconvertible to RDKIT Mols/catch some weird errors
        except Exception as e:
            failed_count += 1 #keep track of how many mols failed
            continue

    #if any mols failed, print how many 
    if failed_count >= 1:
        print(f"\nFailed to convert {failed_count} SMILES to RDKit Mols.")

    #give some feedback on the # of successfully converted Mols
    print(f"\nSucessfully converted {len(list_of_valid_mols)} SMILES to valid RDKit Mols.")    
    
    #return the list of validated RDkit mols
    return list_of_valid_mols

In [4]:
def mols_to_unique_smiles_rdkit(mol_list: List[Chem.Mol]) -> List[str]:
    """
    Function: Converts a list of RDKit mol objects into a list of unique (deduplicated) SMILES strings 
    
    Input:
        - mol_list: List[Chem.Mol]; a list of valid RDKit Mol objects
        
    Returns:
        -List[str]; a returned list of SMILES strings
    """
    
    return list({Chem.MolToSmiles(mol) for mol in mol_list if mol is not None})

In [5]:
def deduplicate_list_of_mols(list_of_mols: List[Chem.Mol]) -> List[Chem.Mol]:
    """
    Function: Given a list of RDKit mol objects, 

    Input:
        -list_of_mols: List[Chem.Mol]; a list of validated RDKit mol objects

    Returns:
        - unique_mols: List[Chem.Mol]; a list of deduplicated and validated RDKit mol objects
            (based on canonical smiles) 
    """

    print(f"\nChecking for any duplicate SMILES following skeletal editing...")
    
    #convert the list of mols into canonical SMILES with RDKit 
    canonical_smiles = [Chem.MolToSmiles(mol) for mol in list_of_mols]
    #print(f"num canonical_smiles = {len(canonical_smiles)}")

    #Loop over the SMILES and add them to a list if they aren't currently in the list
    unique_smiles = []
    [unique_smiles.append(smile) for smile in canonical_smiles if smile not in unique_smiles]

    #print(f"num unique_smiles = {len(unique_smiles)}")
    
    #convert the unique SMILES back into RDKit mols
    unique_mols = [Chem.MolFromSmiles(smi) for smi in unique_smiles]

    num_duplicates = len(canonical_smiles) - len(unique_mols)
    if num_duplicates:
        print(f"\tRemoved {num_duplicates} duplicate SMILES")
        
    elif num_duplicates == 0:
        print(f"\tNo duplicate SMILES following skeletal editing.")

    #return the unique mols
    return(unique_mols)

In [6]:
def remove_exocyclic_phs(list_of_valid_mols: List[Chem.Mol], printing=False) -> List[Chem.Mol]:
    """
    Function: parses validated RDKit mol objects for exocylic Ph substituents (basic Ph attached to a ring)
        using SMARTS-based substructure matching. Replaces any matches with a simple -Me group. Returns a 
        list of new mol objects with any/all exocylic Ph substituents removed. 

    Input:
        - list_of_valid_mols: List[Chem.Mol]; a list of validated RDKit mol objects passed from/by the 
            smiles_to_mols_rdkit() method.
        - printing=False: Default Bool; optionally turn on/off debug printing

    Returns:
        - processed_mols: List[Chem.Mol]: a list of validated RDkit mol objects with any/all exocylic 
            Ph substitutents removed
    """

    #give user some debug info
    print(f"Parsing Mols for and removing any exocylic -Ph substituents (replacing with -Me)...")
    
    #store aliphatic-bound Ph SMARTS for substructure replacement as var
    exocyclic_ph_smarts = Chem.MolFromSmarts('[cR1]1[cD2R1][cD2R1][cD2R1][cD2R1][cD2R1]1')
    
    #replacing aliphatic Ph's with a R-Me group; could be whatever you wanted/needed
    replace_with = Chem.MolFromSmiles('C')
    
    #list to store the edited/non-edited mols 
    processed_mols = []

    #make a kind of dumb counter to see how many scaffolds we've edited
    num_edited = 0

    #loop over each of the mols in the list for substructure matches of our SMARTS pattern
    for mol in list_of_valid_mols:

        #Check for substructure matches matching our SMARTS pattern
        matches = mol.GetSubstructMatches(exocyclic_ph_smarts)

        #match found; apply our SMARTS pattern replacement from above
        if matches: 

            #debug for validation
            if printing:
                print(f"\nFound exocylic Ph(s): {Chem.MolToSmiles(mol)}")

            #Replace one substructure (ring-Ph) with another (ring-Me), replace all instances
            mol_noPh = Chem.ReplaceSubstructs(mol, exocyclic_ph_smarts, replace_with, replaceAll=True)  
            processed_mols.append(mol_noPh[0])

            #debug for validation
            if printing:
                print(f"After Replacement: {Chem.MolToSmiles(mol_noPh[0])}")
                
            num_edited += 1

        #no match was found; mol doesn't contain substructure
        elif not matches: 
            #append the mol; no edits needed
            processed_mols.append(mol) 

    #if any mols were edited, print how many 
    if num_edited != 0:
        print(f"\tNumber of molecular scaffolds edited: {num_edited}")

    #deduplicate the list of mols, just in case
    deduplicated_valid_mols = deduplicate_list_of_mols(list_of_valid_mols)
        
    #return the list of mols with exocylic Ph subs. removed
    return(deduplicated_valid_mols)

In [7]:
def find_vinylic_bond_indices(mol: Chem.Mol, printing=False) -> List[Tuple]:
    """
    Function: Perform a substructure search for exocylic vinylic substituents. Where present, retain
        a list of atom/bond indices corresponding to the vinylic atoms. 

    Input:
        - mol: Chem.Mol; a validated RDKit mol object
        - printing=False: Default Bool; optionally turn on/off debug printing

    Returns:
        - bond_list: List[Tuple]
    """
    
    #init the mol/ring info
    Chem.SanitizeMol(mol)
    
    #SMARTS Pattern to ID vinylic ring-C=CR's
    #"any aromatic atom with 3 explicit connections singly bound to the same"
    vinylic_smarts = Chem.MolFromSmarts('[c,n,s,o,p;R]-[$([CX3]=[CX3])]')

    #check how many SMARTS pattern matches in mol
    init_matches = mol.GetSubstructMatches(vinylic_smarts)
    num_subs = len(init_matches)

    #no matches found; continue
    if num_subs == 0:
        next

    #found a SMARTS match; mol contains at least 1 vinylic group
    elif num_subs >= 1:

        #debug for validation
        if printing:
            print(f"\nFound Vinylic!\nOriginal SMILES: {Chem.MolToSmiles(mol)}")

        #grab matches as tuple
        matches = [x for x in init_matches]

        #conv tup to list of bonds (list of 2 atoms defining a bond)
        bond_list = []
        for match in matches:
            bond_list.append(list(match))

        #return list of vinylic bond indices
        return bond_list

In [8]:
def fragment_mol_on_vinyl_indices(mol: Chem.Mol, bond_list: list[Tuple], printing=False) -> List[Chem.Mol]:
    """
    Function: SMARTS-based Substructure search of RDKit Mol object for exocylic vinyl substitutents. 
        Fragments mol on exocylic bond indices to produce two (or more) subunits. Checks each fragment
        for aromatic components, discards fragment if not aromatic. Returns a list of putatively mols generated 
        from fragments 
    
    Input:
        - mol: Chem.Mol; a validated RDKit mol object
        - bond_list: list[Tuple]; a tuple containing paired vinylic bond indices; passed by the 
            find_vinylic_bond_indices() method, above.
        - printing=False: Default Bool; optionally turn on/off debug printing
            
    Returns:
        - fragmented_mols: List(Chem.Mol); a list of putatively aromatic candidate mols (usually 1) generated
            via fragmentation. Not guaranteed to be unique against external lists of mols, so deduplicate if 
            concatenating the return list to a bigger list of SMILES/Mols.
    """

    #RDKit mols are immutable - they can't be modified
    #we create an Read-Write Molecule (RWMol) from the mol
    rwmol = Chem.RWMol(Chem.RemoveHs(mol))

    #Loop over each of the atom pairs in the list of vinylic indices
    for atom_pair in bond_list:
        #delete the bond between the paired atoms (ring-//-C=CR)
        rwmol.RemoveBond(atom_pair[0], atom_pair[1])

        #replace the chopped Atom between fragments with an H atom
        for atom in atom_pair:
            rwmol.AddAtom(Chem.Atom(1)) #H atom is index 1 in Periodic Table

            #Figure out the index we replaced 
            #maybe dont need this
            H_idx = rwmol.GetNumAtoms() - 1

            #add a single covalent bond between the original atom and its new neighbor H 
            rwmol.AddBond(atom, H_idx, Chem.BondType.SINGLE)

    #grab all of our changes into a new mol object
    new_mol = rwmol.GetMol()

    #Our mol's  is now split into fragments delimited by '.'
    #debug for validation
    if printing:
        #Our mol's  is now split into fragments delimited by '.'
        print(f'Separated Mol: {Chem.MolToSmiles(new_mol)}')

    #split the '.'-joined mol into its fragment mols
    split_frag_indices = Chem.rdmolops.GetMolFrags(new_mol, sanitizeFrags=False)

    #Now the fragments are split, want to see if either of the fragments is still aromatic
    #We want to keep the "good" fragments and abandon the ones without aromatic sections
    fragmented_mols = []
    for frag in split_frag_indices:

        #Here we loop over each ATOM in the mol fragment and check if they are aromatic
        for a in rwmol.GetAtoms():
            #set ATOMS that arent in a ring and aromatic to "NOT AROMATIC" 
            if (not a.IsInRing()) and a.GetIsAromatic():
                a.SetIsAromatic(False)
                
        #Here we loop over each BOND in the mol fragment and check if they are aromatic        
        for b in rwmol.GetBonds():
            #set BONDS that arent in a ring and aromatic to "NOT AROMATIC"
            if (not b.IsInRing()) and b.GetIsAromatic():
                b.SetIsAromatic(False)

        new_mol = rwmol.GetMol()
        mol_frag = Chem.MolFromSmiles(Chem.MolFragmentToSmiles(rwmol, frag, allHsExplicit=True))

        #check if either/any fragment is aromatic/contains an aromatic subgroup
        aro_flags = Chem.MolFromSmarts('[c,n,s,o,p;R]')
        aro_matches = mol_frag.GetSubstructMatches(aro_flags) #apply our SMARTS check to the frag

        #if the fragment contains aromatic component(s), 
        if aro_matches: #if the fragment contains aromatic component(s),

            #debug for validation
            if printing:
                print(f"Found aromatic fragment: {Chem.MolToSmiles(mol_frag)}")
            
            #Convert the aromatic mol to a SMILES string for dedup.
            smi_string = Chem.MolToSmiles(mol_frag)
            
            fragmented_mol = Chem.MolFromSmiles(smi_string)

            fragmented_mols.append(fragmented_mol)
            
    if fragmented_mols:
        
        return(fragmented_mols)

In [9]:
def find_biaryl_sigma_bond_indices(mol: Chem.Mol, printing=False) -> List[Tuple]:
    """
    Function: Perform a substructure search for exocylic vinylic substituents. Where present, retain
        a list of atom/bond indices corresponding to the vinylic atoms. 

    Input:
        - mol: Chem.Mol; a validated RDKit mol object
        - printing=False: Default Bool; optionally turn on/off debug printing

    Returns:
        - bond_list: List[Tuple]
    """

    #init the mol/ring info
    Chem.SanitizeMol(mol)
    
    #SMARTS to ID's biaryls
    biaryl_smarts = Chem.MolFromSmarts('[aD3;R1]-[aD3;R1]')

    #check how many SMARTS pattern matches in mol
    init_matches = mol.GetSubstructMatches(biaryl_smarts)
    num_subs = len(init_matches)

    #no matches found; continue
    if num_subs == 0: 
        next

    #found a SMARTS match; mol contains at least 1 biaryl group
    elif num_subs >= 1:
        
        #debug for validation
        if printing:
            print(f"\nFound Biaryl!\nOriginal SMILES: {Chem.MolToSmiles(mol)}")
        
        #grab matches as tuple
        matches = [x for x in init_matches]
        
        #conv tup to list of bonds (list of 2 atoms defining a bond)
        bond_list = []
        for match in matches:
            bond_list.append(list(match))

        #return list of biaryl sigma bond indices
        return bond_list

In [10]:
def fragment_mol_on_biaryl_indices(mol: Chem.Mol, bond_list: list[Tuple], printing=False) -> List[Chem.Mol]:
    """
    Function: SMARTS-based Substructure search of RDKit Mol object for exocylic vinyl substitutents. 
        Fragments mol on exocylic bond indices to produce two (or more) subunits. Checks each fragment
        for aromatic components, discards fragment if not aromatic. Returns a list of putatively mols generated 
        from fragments 
    
    Input:
        - mol: Chem.Mol; a validated RDKit mol object
        - bond_list: list[Tuple]; a tuple containing paired vinylic bond indices; passed by the 
            find_biaryl_sigma_bond_indices() method, above.
        - printing=False: Default Bool; optionally turn on/off debug printing
            
    Returns:
        - fragmented_mols: List(Chem.Mol); a list of putatively aromatic candidate mols (usually 1) generated
            via fragmentation. Not guaranteed to be unique against external lists of mols, so deduplicate if 
            concatenating the return list to a bigger list of SMILES/Mols.
    """

    #RDKit mols are immutable - they can't be modified
    #we create an Read-Write Molecule (RWMol) from the mol
    rwmol = Chem.RWMol(Chem.RemoveHs(mol))

    #Loop over each of the atom pairs in the list of vinylic indices
    for atom_pair in bond_list:
        #delete the bond between the paired atoms (ring-//-ring)
        rwmol.RemoveBond(atom_pair[0], atom_pair[1])

        #replace the chopped Atom between fragments with an H atom
        for atom in atom_pair:
            rwmol.AddAtom(Chem.Atom(1)) #H atom is index 1 in Periodic Table

            #Figure out the index we replaced 
            #maybe dont need this
            H_idx = rwmol.GetNumAtoms() - 1

            #add a single covalent bond between the original atom and its new neighbor H 
            rwmol.AddBond(atom, H_idx, Chem.BondType.SINGLE)

    #grab all of our changes into a new mol object
    new_mol = rwmol.GetMol()

    #Our mol's  is now split into fragments delimited by '.'
    #debug for validation
    if printing:
        #Our mol's  is now split into fragments delimited by '.'
        print(f'Separated Mol: {Chem.MolToSmiles(new_mol)}')

    #split the '.'-joined mol into its fragment mols
    split_frag_indices = Chem.rdmolops.GetMolFrags(new_mol, sanitizeFrags=False)

    #Now the fragments are split, want to see if either of the fragments is still aromatic
    #We want to keep the "good" fragments and abandon the ones without aromatic sections
    fragmented_mols = []
    for frag in split_frag_indices:

        #Here we loop over each ATOM in the mol fragment and check if they are aromatic
        for a in rwmol.GetAtoms():
            #set ATOMS that arent in a ring and aromatic to "NOT AROMATIC" 
            if (not a.IsInRing()) and a.GetIsAromatic():
                a.SetIsAromatic(False)
                
        #Here we loop over each BOND in the mol fragment and check if they are aromatic        
        for b in rwmol.GetBonds():
            #set BONDS that arent in a ring and aromatic to "NOT AROMATIC"
            if (not b.IsInRing()) and b.GetIsAromatic():
                b.SetIsAromatic(False)

        new_mol = rwmol.GetMol()
        mol_frag = Chem.MolFromSmiles(Chem.MolFragmentToSmiles(rwmol, frag, allHsExplicit=True))

        #check if either/any fragment is aromatic/contains an aromatic subgroup
        aro_flags = Chem.MolFromSmarts('[c,n,s,o,p;R]')
        aro_matches = mol_frag.GetSubstructMatches(aro_flags) #apply our SMARTS check to the frag

        #if the fragment contains aromatic component(s), 
        if aro_matches: #if the fragment contains aromatic component(s),

            #debug for validation
            if printing:
                print(f"Found aromatic fragment: {Chem.MolToSmiles(mol_frag)}")
            
            #Convert the aromatic mol to a SMILES string for dedup.
            smi_string = Chem.MolToSmiles(mol_frag)
            
            fragmented_mol = Chem.MolFromSmiles(smi_string)

            fragmented_mols.append(fragmented_mol)
            
    if fragmented_mols:
        
        return(fragmented_mols)

In [11]:
def remove_exoclic_substituents(list_of_mols: List[Chem.Mol], printing=False) -> List[Chem.Mol]:
    """
    Function: Parses a list of RDKit mol objects to find any exocylic substituents (things attached to the core
    rings of cyclic heteroaromatic mols) - finds these substituents and replaces them with H. Ignores ring-embedded
    ketones and imines. Returns a list of unique (deduplicated) mols. 
    
    Input: 
        - list_of_mols: List[Chem.Mol]; a list of pre-parsed RDKit Mol objects corresponding to cyclic arene cores 
        - printing=False: Default Bool; optionally turn on/off debug printing
        
    Returns:
        - list_of_mols: List[Chem.Mol]; a list of RDKit Mol objects for arene cores with all substituents removed 
    """

    #SMARTS pattern for non-ring R groups (what we want to remove from each core)
    r_smarts = Chem.MolFromSmarts('[*!$([O]=[cX3])&!$([N]=[cX3]);!R,!a]') #ignoring cyclic ketones/imines 
    
    #convert canon_smiles to list of mols
    #mols = [Chem.MolFromSmiles(smile) for smile in list_of_smiles]

    #now we loop over our mols and apply our truncation algorithm from above
    truncated_mols = []

    #init a counter to keep track of number of cores
    num_added = 0

    #loop over each of the mols in prepared list of mols 
    for mol in list_of_mols:

        #debug printing
        if printing: 
            print(f'\nSMILES prior to chop: {Chem.MolToSmiles(mol)}')

        #create a read-write-mol for skeletal editing
        prepped_mol = Chem.RWMol(Chem.RemoveHs(mol))

        #search the RWMol for substructure matches (any exocylic R group other than ketones/imines)
        matches = prepped_mol.GetSubstructMatches(r_smarts)
        matches = [x[0] for x in matches]
        #print(f'\nAtom indices to be removed: {matches}')

        #we also want the complementary list of atom indices that we want to retain (everything in a ring)
        not_selected = [x.GetIdx() for x in prepped_mol.GetAtoms() if x.GetIdx() not in matches]
        rings = prepped_mol.GetRingInfo()
        rings.AtomRings()
        atoms_of_interest = not_selected

        modification_bonds = []
        for atom in prepped_mol.GetAtoms():
            if atom.GetIdx() in atoms_of_interest: #if our atom index is in the ring,
                
                neighbors = atom.GetNeighbors() #find everything the ring atom is connected to as tuple
                
                for neighbor in neighbors: #consider each atom connected to our ring atom of interest
                    #if the connected atom is not H and the connected atom isn't in our ring,
                    if neighbor.GetSymbol()!='H' and neighbor.GetIdx() not in atoms_of_interest:
                        #we want to change that bond - this is finding/storing every c/n/s-[X] bond by atom index
                        modification_bonds.append([atom.GetIdx(),neighbor.GetIdx()])

        #sanity check our bonds to be chopped (compare against the images, above, for indexing)
        #print(f'R-group bond indices to be chopped: {modification_bonds}')

        #loop over the list of located bonds to remove
        for bond in modification_bonds:
            #print('Modifyting Bond ', bond) #debug
        
            #remove the indexed bond from the molecule
            prepped_mol.RemoveBond(bond[0],bond[1])
        
            #add in a new H atom in place
            prepped_mol.AddAtom(Chem.Atom(1)) 
            
            #get the atom index of the newly added H
            H_idx = prepped_mol.GetNumAtoms()-1
            
            #appending that H index ensures that the newly added H is retained
            atoms_of_interest.append(H_idx)
            
            # print('Adding Bond ',(bond[0], C_idx))
            prepped_mol.AddBond(bond[0], H_idx, Chem.BondType.SINGLE)

        new_mol = prepped_mol.GetMol()

        #we now have a list of all the atom indices we want to retain and a list of everything we don't want
        #print(f'\033[1mCore atoms to retain: {atoms_of_interest}\n\033[0m')

        #apply the chops - of the original mol, retain only the atom indices we want
        truncated_core_mol = Chem.MolFromSmiles(Chem.MolFragmentToSmiles(new_mol, atoms_of_interest))

        # #can see some molecules generate fragments after chopping non-ring substituents; have to check explicitly
        # if printing:
        #     print(f"Top bit truncated_core_mol = {Chem.MolToSmiles(truncated_core_mol)}")
        
        #new_mol = truncated_core_mol.GetMol()
        #print(f'Truncated Core: {Chem.MolToSmiles(truncated_core_mol)}\n')

        #check if the mol has fragments
        frags_after_truncation = Chem.rdmolops.GetMolFrags(truncated_core_mol, sanitizeFrags=True)
        num_frags = len(frags_after_truncation)
        #print(num_frags)

        #if there are fragments:
        if num_frags == 1: #if a single fragment;
            try:
                Chem.SanitizeMol(truncated_core_mol) #try to sanitize the OG mol

                if printing:
                    print(f"Truncated Core: {Chem.MolToSmiles(truncated_core_mol)}")

                num_added += 1
                
                #append the truncated mol to the list of truncated mols:
                truncated_mols.append(truncated_core_mol)

            except:
                #print("failed")
                pass

        elif num_frags > 1: 
            #print("found multiple fragments")
            for fragment in frags_after_truncation:
                try:
                    truncated_frag = Chem.MolFromSmiles(Chem.MolFragmentToSmiles(truncated_core_mol, fragment))

                    if printing:
                        print(f"Truncated Core: {Chem.MolToSmiles(truncated_frag)}")

                    num_added += 1
                    
                    truncated_mols.append(truncated_frag)

                except:
                    #print("failed")
                    pass
                    
        #reset lists between mols
        matches = []
        atoms_of_interest = []
        
    #apply a rigorous set-based deduplication of SMIs
    arene_core_smis = mols_to_unique_smiles_rdkit(truncated_mols)

    #debug on the number of arene cores removed due to duplication
    if printing:
        num_removed = num_added - len(arene_core_smis)
        print(f"\nTruncation resulted in {num_added} potential mols before deduplication.")
        print(f"\nDeduplication removed {num_removed} duplicate mols")

    #return deduplicated smiles as a list of mols
    deduplicated_truncated_mols = smiles_to_mols_rdkit(arene_core_smis)

    return deduplicated_truncated_mols

In [12]:
def confirm_mols_contain_internal_alkene(list_of_mols: List[Chem.Mol]) -> List[Chem.Mol]:
    """
    Function:

    Input:
        - list_of_mols: List[Chem.Mol]; a list of RDKit Mol objects corresponding to truncated aromatic cores
    Returns:
        - confirmed_alkenes: List[Chem.Mol]; a list of RDKit Mol objects confirmed to contain at least one internal
            C=C bond that is convertible to a C#C bond.   
        - no_alkenes: List[Chem.Mol]; a list of RDKit Mol objects confirmed to NOT contain an internal alkene
    """


    #SMARTS to match on
    smarts_def = '[cH1D2;R][cH1D2;R]'
    alkene_smarts = Chem.MolFromSmarts(smarts_def)
    
    #loop over mols, retain only ones with internal alkenes
    confirmed_alkenes = []
    no_alkenes = []

    for mol in list_of_mols:
        matches = mol.GetSubstructMatches(alkene_smarts)

        if not matches:
            no_alkenes.append(mol)
        else:
            confirmed_alkenes.append(mol)

    return confirmed_alkenes, no_alkenes
    

In [13]:
""" Convert SMILES data from CSV to RDKit Mol objects

This cell parses the incoming_data.csv file for a column named 'smiles' and attempts to convert
each SMILES string to a valid/sanitized RDKit Mol object. It will return a list of mols stored to
the "validated_mols" variable. 
"""

#Convert SMILES data in the incoming_data.csv to RDKit Mols
validated_mols = smiles_to_mols_rdkit(incoming_data.smiles)


Sucessfully converted 29918 SMILES to valid RDKit Mols.


In [14]:
""" SMARTS substucture search to find and remove Exocyclic Phenyl substituents (replace with ring-Me)

This cell parses a list of RDKit mols for exocylic Ph substituents (ring-Ph) using SMARTS-based
substructure search for each mol. If matches are found (structure has ring-Ph substituent(s)), 
slice the ring-Ph bond and replace with ring-Me. Deduplicate the resulting list of mols. 

Optionally set "printing=True" to show output for each mol if matches are found. 
"""

#SMARTS substucture search to remove Exocyclic Phenyl substituents (replace with r-Me)
no_ph_mols = remove_exocyclic_phs(validated_mols, printing=False)

print(f"\nCurrent number of candidate SMILES following parsing: {len(no_ph_mols)}")

Parsing Mols for and removing any exocylic -Ph substituents (replacing with -Me)...
	Number of molecular scaffolds edited: 290

Checking for any duplicate SMILES following skeletal editing...
	No duplicate SMILES following skeletal editing.

Current number of candidate SMILES following parsing: 29918


In [15]:
""" SMARTS-based substructure search to find ring-bound vinylic groups

This cell takes the previous cell's list of RDKit Mols (with all -Ph's removed) and checks for
vinylic groups connected to aromatic rings. If found, the SMILES string is fragmented on the C=C
bond to generate unique fragments. Parse each of the generated fragments for aromatic subgroups 
and retain any fragments that have an aromatic core. Deduplicate the set to retain a list of 
unique aromatic cores without R-Ph or R-C=C groups. 
"""

#give user some debug info
print(f"Parsing {len(no_ph_mols)} Mols for and fragmenting on exocylic ring-C=C (vinylic) substituents...")

#init a list to store parsed mols
no_vinylic_mols = []

#counter to keep track of how many SMILES contained a vinylic ring substitutent (Ring-C=CR)
vinylic_count = 0
new_frag_count = 0

#loop over each of the mols in current list of valid mols
for mol in no_ph_mols:
    
    #substructure search for vinylic bond indices; if populated, some have been found 
    vinylic_bond_indices = find_vinylic_bond_indices(mol, printing=False)

    #If no vinylic indices found, mol doesn't contain vinylic subgroup
    if not vinylic_bond_indices:
        no_vinylic_mols.append(mol) #retain the good mols to the list of mols

    #if there are bonds to break, fragment them on vinylic bond
    elif vinylic_bond_indices:

        #increment the vinylic counter
        vinylic_count += 1

        #split the mol into fragments at the ring-//-C=CR bond; set printing=False to silence
        chopped_mols = fragment_mol_on_vinyl_indices(mol, vinylic_bond_indices, printing=False)

        for chopped_mol in chopped_mols:
            new_frag_count += 1
            no_vinylic_mols.append(chopped_mol)

#debug for info on operations
print(f"\nFound {vinylic_count} candidate SMILES featuring an exoclyic vinylic substitutent.")
print(f"\nFragmentation produced {new_frag_count} 'new' aromatic SMILES before deduplication.")

#Deduplicate the working list of mols now that ring-C=Cs have been fragmented/parsed. 
deduplicated_valid_mols = deduplicate_list_of_mols(no_vinylic_mols) 

print(f"\nCurrent number of candidate SMILES following parsing: {len(deduplicated_valid_mols)}")

Parsing 29918 Mols for and fragmenting on exocylic ring-C=C (vinylic) substituents...

Found 85 candidate SMILES featuring an exoclyic vinylic substitutent.

Fragmentation produced 101 'new' aromatic SMILES before deduplication.

Checking for any duplicate SMILES following skeletal editing...
	Removed 77 duplicate SMILES

Current number of candidate SMILES following parsing: 29857


In [16]:
""" SMARTS-based substructure search to find and separate on bi/multi-aryls 

This cell takes the previous cell's list of RDKit Mols (with all -Ph's/-C=CR's removed) and checks for
multiple distinct aryl groups separated by single bonds (joined ring systems). If found, the SMILES
string is fragmented on the biaryl sigma bond to generate unique aryl fragments. 

Parse each of the generated fragments for aromatic subgroups 
and retain any fragments that have an aromatic core. Deduplicate the set to retain a list of 
unique aromatic cores without R-Ph or R-C=C groups. 
"""

#give user some debug info
print(f"Parsing {len(deduplicated_valid_mols)} Mols for distinct aryl sub-units bound by a sigma bond and fragmenting...")

#init a list to store parsed mols
no_biaryl_mols = []

#counter to keep track of how many SMILES contained a vinylic ring substitutent (Ring-C=CR)
biaryl_count = 0
new_biaryls_count = 0

#loop over each of the mols in current list of valid mols
for mol in deduplicated_valid_mols:
    
    #substructure search for vinylic bond indices; if populated, some have been found 
    biaryl_bond_indices = find_biaryl_sigma_bond_indices(mol, printing=False)

    #If no vinylic indices found, mol doesn't contain vinylic subgroup
    if not biaryl_bond_indices:
        no_biaryl_mols.append(mol) #retain the good mols to the list of mols

    #if there are bonds to break, fragment them on vinylic bond
    elif biaryl_bond_indices:

        #increment the vinylic counter
        biaryl_count += 1

        #split the mol into fragments at the ring-//-C=CR bond; set printing=False to silence
        chopped_mols = fragment_mol_on_biaryl_indices(mol, biaryl_bond_indices, printing=False)

        for chopped_mol in chopped_mols:
            new_frag_count += 1
            no_biaryl_mols.append(chopped_mol)

#debug for info on operations
print(f"\nFound {biaryl_count} candidate Mols featuring an exoclyic vinylic substitutent.")
print(f"\nFragmentation produced {new_frag_count} 'new' aromatic Mols before deduplication.")

#Deduplicate the working list of mols now that ring-C=Cs have been fragmented/parsed. 
deduplicated_pre_trunc_mols = deduplicate_list_of_mols(no_biaryl_mols) 

#Add some diagnostics on what was done
print(f"\nCurrent number of candidate Mols following parsing: {len(deduplicated_pre_trunc_mols)}")

Parsing 29857 Mols for distinct aryl sub-units bound by a sigma bond and fragmenting...

Found 433 candidate Mols featuring an exoclyic vinylic substitutent.

Fragmentation produced 1005 'new' aromatic Mols before deduplication.

Checking for any duplicate SMILES following skeletal editing...
	Removed 768 duplicate SMILES

Current number of candidate Mols following parsing: 29560


In [17]:
""" Parse for and Truncate Exocylic Substituents - Retain [hetero]cyclic Arene Cores

This cell operates on a list of mols from the previous cell, deduplicated_pre_trunc_mols. These mols
have been parsed for exocyclic -Ph, -C=CR groups and ensured to not consist of biaryl systems. The code
below will parse each mol for all of its remaining exocyclic substituents and remove them, replacing them
with a H atom. The mols are sanitized and deduplicated and stored in the "truncated_mols" variable. 
"""

#give user some debug info
print(f"Parsing for and removing exocyclic substituents from {len(deduplicated_pre_trunc_mols)} Mols...")

#this method chops all exocyclic substituents and returns a list of (unique/deduplicated) chopped mols
truncated_mols = remove_exoclic_substituents(deduplicated_pre_trunc_mols, printing=False)

#Add some diagnostics on what was done
print(f"\nRetained {len(truncated_mols)} unique [hetero]aromatic cores as RDKit Mols.")

Parsing for and removing exocyclic substituents from 29560 Mols...

Sucessfully converted 24887 SMILES to valid RDKit Mols.

Retained 24887 unique [hetero]aromatic cores as RDKit Mols.


In [18]:
""" Confirm candidate Mols contain at least one internal Alkene (C=C in a ring)

This cell performs a SMARTS-based substructure search for an internal (within-the-ring) C=C double bond. 
The double bond needs to be convertible to a C#C triple bond. We retain mols that do and do not meet this
criteria, as many of the structures in the dataset are of potential interest but don't meet our criteria. 
Once classified, the resulting lists of mols are converted to lists of canonical SMILES for writing to a
two separate .csv files. 
"""

#give user some debug info
print(f"Substructure searching {len(truncated_mols)} RDKit Mols for at least one convertible C=C in the ring...")

confirmed_alkene_mols, no_alkene_mols = confirm_mols_contain_internal_alkene(truncated_mols)

#Add some diagnostics on what was done
print(f"\nNumber of heteroaromatic scaffolds confirmed to feature at least one convertible C=C in the ring: {len(confirmed_alkene_mols)}.")
print(f"\nNumber of heteroaromatic scaffolds without a convertible C=C in the ring: {len(no_alkene_mols)}.")

#convert the mols to lists of canonical SMILES
confirmed_arene_smiles = mols_to_unique_smiles_rdkit(confirmed_alkene_mols)
no_arene_smiles = mols_to_unique_smiles_rdkit(no_alkene_mols)

Substructure searching 24887 RDKit Mols for at least one convertible C=C in the ring...

Number of heteroaromatic scaffolds confirmed to feature at least one convertible C=C in the ring: 8581.

Number of heteroaromatic scaffolds without a convertible C=C in the ring: 16306.


In [19]:
""" ### Output SMILES to CSV for Confirmed C=C-containing [Hetero]Aromatic Cores ### 

This cell writes SMILES data to a formatted .csv. The SMILES data correspond to heteroaromatic cores
featuring at least one C=C double bond convertible to a C#C triple bond. A second round of deduplication
via Pandas string deduplication of SMILES is applied as a last sanity check before generating Arynes.
"""

#create a Pandas DataFrame to store our newly generated Arene Core SMILES data
deduplicated_cores = pd.DataFrame(confirmed_arene_smiles, columns=['arene_smiles'])

#just in case deduplication with Pandas
deduplicated_cores.drop_duplicates(subset='arene_smiles', inplace=True)
arenes_final = deduplicated_cores.reset_index(drop=True)

#gen a new ID for each of the dedup'ed cores
arene_ids = []
i = 1
for core in confirmed_arene_smiles:
    core_id = f'arene_{i}' 
    arene_ids.append(core_id)
    i+=1
    
# #debug    
# #check the length of this; should be 1:1
# print(f'Num Cores: \t{len(final_cores)}')
# print(f'Num IDS: \t{len(arene_ids)}')

#add IDs to a new df
arene_df = pd.DataFrame(arene_ids, columns=['arene_ID'])

#add the deduplicated arene core smiles data as a new col
arene_df['arene_smiles'] = confirmed_arene_smiles

# #debug checking
# arene_df.head()

#output the arene DF as a .csv
csv_name = module2_dir / 'confirmed_arene_cores.csv'
arene_df.to_csv(csv_name, index=False)
print(f"\033[1m'{csv_name.name}' containing {len(arene_df)} arene SMILES written '/{module2_dir.name}/'.\033[0m")

'confirmed_arene_cores.csv' containing 8581 arene SMILES written '/Module2_Prepare_Arenes/'.


In [20]:
""" ### OPTIONAL: Output SMILES to CSV for [Hetero]Aromatic Cores that CANNOT be Converted to Arynes ### 

This cell writes SMILES data to a "non-convertible_cores.csv" file for mols that have passed all rounds of 
triage but don't contain an internal C=C and thus can't be converted to a valid C#C-containing aryne. However,
these SMILES are of general interest, potentially, so we will output the parsed SMILES, regardless. 
"""

#create a Pandas DataFrame to store our newly generated Arene Core SMILES data
deduplicated_invalids = pd.DataFrame(no_arene_smiles, columns=['no_arene_smiles'])

#just in case deduplication with Pandas
deduplicated_invalids.drop_duplicates(subset='no_arene_smiles', inplace=True)
invalids_final = deduplicated_invalids.reset_index(drop=True)

#gen a new ID for each of the dedup'ed cores
bad_arene_ids = []
i = 1
for core in no_arene_smiles:
    core_id = f'no_arene_{i}' 
    bad_arene_ids.append(core_id)
    i+=1
    
#add IDs to a new df
bad_arene_df = pd.DataFrame(bad_arene_ids, columns=['bad_hetero_ID'])

#add the deduplicated arene core smiles data as a new col"
bad_arene_df['no_arene_smiles'] = no_arene_smiles

#output the arene DF as a .csv
csv_name = module2_dir / 'Non-convertible_cores.csv'
bad_arene_df.to_csv(csv_name, index=False)
print(f"\033[1m'{csv_name.name}' containing {len(bad_arene_df)} arene SMILES written '/{module2_dir.name}/'.\033[0m")

'Non-convertible_cores.csv' containing 16306 arene SMILES written '/Module2_Prepare_Arenes/'.
